<!-- Professional, seductive header, Candy AI / Midnight Dark style -->
<div style="background: linear-gradient(90deg, #1a0022 0%, #39005c 60%, #ff1b6b 100%); border-radius: 30px; margin-bottom: 0.5em; box-shadow: 0 8px 48px 0 #210029b0; padding: 2em 1.5em 1em 1.5em; border: 2px solid #9B50D0;">
  <h1 style="font-family: 'Outfit', 'Montserrat', 'Rubik', Arial, sans-serif; font-size: 2.7em; color: #fff; margin-bottom: 0.2em; font-weight: 800; letter-spacing: 0.01em; text-shadow: 0 2px 24px #d144a7b0;">
    🔥 <span style="color:#FE1AAD">AI Porn Video Studio</span>
  </h1>
  <div style="font-size:1.25em; color:#e9acf6; font-family: 'Montserrat', 'Outfit', Arial,sans-serif; font-weight:500;">Ultra Fast, Stunning NSFW Text-to-Video: <span style='color:#ff70b6;'>Q4</span>/<span style='color:#d7b5ff;'>Q5</span>/<span style='color:#af36ff;'>Q6</span> Modes · LoRA Blending · Speed Presets · Candy/UI</div>
</div>


In [ ]:
# @title 🍑 FAST T2V NSFW Generator — Gorgeous Modern Gradio UI

def generate_video(
    positive_prompt: str, negative_prompt: str,
    width: int, height: int, seed: int, steps: int, cfg_scale: float,
    sampler_name: str, scheduler: str, frames: int, fps: int,
    lora_name: str = "", lora_strength: str = "1.0", model_quant: str = "Q5",
    output_format: str = "mp4", ultra_fast_mode: bool = False
):
    if ultra_fast_mode:
        width, height, frames, steps, cfg_scale, model_quant = 480, 832, 8, 12, 1.0, "Q4"
    if seed == -1:
        seed = random.randint(0, 2**63 - 1)
        print(f"[INFO] Random seed chosen: {seed}")
    with torch.inference_mode():
        print("[INFO] Loading Text Encoder...")
        clip = clip_loader.load_clip("umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default")[0]
        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]
        clear_memory()
        empty_latent = empty_latent_video.generate(width, height, frames, 1)[0]
        clear_memory()
        print(f"[INFO] Loading Unet Model ({model_quant}) ...")
        if model_quant == "Q4":
            unet_path = "wan2.1-t2v-14b-Q4_K.gguf"
        elif model_quant == "Q6":
            unet_path = "wan2.1-t2v-14b-Q6_K.gguf"
        else:
            unet_path = "wan2.1-t2v-14b-Q5_0.gguf"
        model = unet_loader.load_unet(unet_path)[0]
        applied_loras = []
        if lora_name.strip():
            names = [n.strip() for n in lora_name.split(',') if n.strip()]
            strengths = [float(x.strip()) for x in lora_strength.replace(',', ' ').split()]
            if len(strengths) < len(names):
                strengths += [strengths[-1]] * (len(names)-len(strengths))
            for l_name, l_strength in zip(names, strengths):
                lora_path = os.path.join('/content/ComfyUI/models/lora', l_name)
                if not os.path.exists(lora_path):
                    print(f"[WARN] Could not find LoRA: {lora_path}. Skipping.")
                    continue
                model = lora_loader.load_lora(model, lora_path, l_strength)[0]
                applied_loras.append(l_name)
            if not applied_loras:
                print('[WARN] No LoRA(s) actually loaded!')
        clear_memory()
        print("[INFO] Sampling video latents...")
        sampled = ksampler.sample(
            model=model, seed=seed, steps=steps, cfg=cfg_scale,
            sampler_name=sampler_name, scheduler=scheduler, positive=positive, negative=negative, latent_image=empty_latent
        )[0]
        del model; clear_memory()
        print("[INFO] Loading VAE and decoding latents...")
        vae = vae_loader.load_vae("wan_2.1_vae.safetensors")[0]
        decoded = vae_decode.decode(vae, sampled)[0]
        del vae; clear_memory()
        output_path = ""
        if frames == 1:
            print("[INFO] Saving single PNG image...")
            output_path = save_as_image(decoded[0], "ComfyUI")
        else:
            if output_format.lower() == "webm":
                print("[INFO] Saving as WEBM...")
                output_path = save_as_webm(decoded, "ComfyUI", fps=fps, codec="vp9")
            elif output_format.lower() == "mp4":
                print("[INFO] Saving as MP4...")
                output_path = save_as_mp4(decoded, "ComfyUI", fps)
            else:
                raise ValueError(f"Unsupported output format: {output_format}")
        clear_memory(verbose=True)
        return output_path

# --- Premium/CandyAI Default Prompts ---
default_positive = (
    "ultra high res, explicit uncensored, drop-dead gorgeous pornstar, glassy eyes, neon photo studio, "
    "lush curves, creamy perfect skin, luscious lips, erotically posed, seductive, glistening, photoreal, "
    "sweaty, sparkly, deep focus, 8k, anime pop, stunning body, camera flash, cumshot, realism"
)
default_negative = (
    "censored, watermark, worst quality, ugly, deformed body, jpeg, extra limbs, mutilated, blurry, cropped, "
    "jpeg artifacts, mosaic, cartoon, poorly drawn, signature, artist name, missing finger, bad composition, poorly rendered genitals"
)

# --- NSFW Preset Prompt Buttons ---
NSFW_PRESETS = {
    "Solo Girl": "solo glamour nude, stunning pornstar, soft lighting, bedroom, erotic pose, tight body, seductive eyes",
    "Doggy Style": "doggy style, sexy nude couple, ass up, wet skin, full view, beautiful pornstar, high res, penetration",
    "Blowjob": "blowjob, beautiful nude girl sucking cock, facial, tongue out, glossy lips, closeup, intense expression",
    "Riding": "cowgirl, riding sex, bouncing tits, energetic, beautiful pornstar, nude, moaning, POV",
    "Lesbian": "lesbian, beautiful naked girls, tongue kissing, groping, intimacy, lingerie, soft touch, wet skin",
    "Ahegao": "ahegao, tongue out, crazy eyes, drooling, orgasm face, hentai, exaggerated expression, cumshot, blushing, ultra high res",
    "Creampie": "creampie, cum dripping, deep penetration, perfect pussy, closeup, beautiful nude pornstar, highly detailed, erotic"
}

# --- Speed Presets ---
SPEED_PRESETS = {
    "Ultra Fast": dict(
        model_quant="Q4", width=480, height=832, frames=8, steps=12, cfg_scale=1.0, sampler_name="uni_pc", scheduler="simple", ultra_fast_mode=True),
    "Balanced": dict(
        model_quant="Q6", width=480, height=832, frames=10, steps=18, cfg_scale=1.1, sampler_name="uni_pc", scheduler="simple", ultra_fast_mode=False),
    "High Quality": dict(
        model_quant="Q5", width=640, height=1024, frames=10, steps=22, cfg_scale=1.2, sampler_name="uni_pc", scheduler="simple", ultra_fast_mode=False)
}

def speed_preset_apply(preset):
    params = SPEED_PRESETS.get(preset, SPEED_PRESETS['Ultra Fast'])
    return (
        params['model_quant'], params['width'], params['height'], params['frames'], params['steps'],
        params['cfg_scale'], params['sampler_name'], params['scheduler'], params['ultra_fast_mode']
    )

# --- Custom CSS for CandyAI/Premium look ---
premium_css = '''
body, .gr-block, .gr-root, .gradio-container {background: #120018 !important; font-family: 'Outfit', 'Montserrat', 'Rubik', Arial, sans-serif;}
.gr-block, .gr-group, .gr-root, .gr-panel, .container, .gradio-container {background:transparent !important;}
.gradio-container {box-shadow:none !important;}
.gr-block, .gr-tab, .gr-accordion, .gr-group, .gr-column, .gr-form, .gr-box, .gr-panel {border-radius: 22px !important; background:#210029aa !important;}
.gr-compact .gr-block {border-radius:22px !important;}
.gradio-card, .gr-card {border-radius:18px !important; background: #18001e9d !important; border:2px solid #50107b52 !important;}
.gr-input, .gr-textbox, .gr-text-input, input, textarea, select, .gr-dropdown, .gr-number {border-radius: 20px !important; background: #2a003b !important; color:#fff !important; font-size: 1.14em !important; border:1.5px solid #8f27d1d9 !important;}
.gr-button, button, .gr-button-lg {border-radius:24px !important; font-size:1.35em !important; font-weight:600; background: linear-gradient(90deg, #ff1b6b 15%, #520074 100%) !important; color:#fff !important; box-shadow:0 0px 20px #50107bcc !important; transition: all 0.18s; padding:14px 38px !important; border: none !important;}
.gr-button:hover, button:hover, .gr-button-lg:hover {background:linear-gradient(97deg,#e100ff 5%,#ff1b6b 95%) !important; transform:scale(1.04) !important; box-shadow:0 0px 36px #d144a777 !important;}
.gr-video {border-radius: 18px !important; overflow: hidden; border: 2.7px solid #9B50D0 !important; box-shadow:0 0px 32px #7814a1a0 !important;}
.gr-slider input[type=range]::-webkit-slider-thumb {background:#ff1b6b !important;}
.gr-slider .gr-box {background: #18001e9a !important;}
.gr-tabs, .gr-tab {background:#210029cc !important; border-radius:19px !important;}
.gr-tab.selected {background:#2c063a !important;}
.gr-accordion {border: none !important;}
.gr-dropdown, .gr-number, .gr-checkbox {border-radius:20px !important;}
::-webkit-scrollbar {width:8px;background:#251637;}
::-webkit-scrollbar-thumb {background:#5f257b; border-radius:7px;}
.gr-markdown {color: #ffe7f8 !important; font-size:1.12em;}
.gradio-container h1, h2, h3, .gradio-container label {font-family:'Outfit','Montserrat','Rubik',Arial,sans-serif;}
.gradio-container h1 {font-size:2.2em; font-weight:800;color:#ff6fd8;text-shadow:0 2px 16px #ff59a6c8;}
.gradio-container label {font-size: 1.11em !important; color:#dec1ff !important; font-weight: 600;}
.gr-form {background: none !important;}
'''

with gr.Blocks(
    title="🔥 AI Porn Video Studio - Luxury Fast Generator",
    css=premium_css
) as demo:
    
    with gr.Tabs(selected=0):
        # -------- Prompt Studio TAB -------- #
        with gr.Tab(label="💋 Prompt Studio"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown('<div style="font-size:1.18em;color:#ff98e5;font-weight:600;margin-bottom:0.7em">Enter your sexiest prompt, or tap a preset below:</div>')
            prompt_box = gr.Textbox(
                label="Positive Prompt",
                value=default_positive,
                lines=5, max_lines=8, min_width=490, show_copy_button=True,
                elem_id="main_positive_prompt"
            )
            # Preset prompt buttons as neat pill buttons
            with gr.Row(equal_height=True):
                preset_btns = []
                for preset, preset_text in NSFW_PRESETS.items():
                    btn = gr.Button(
                        preset,
                        value=preset,
                        elem_id=f"preset_{preset.replace(' ','_').lower()}",
                        interactive=True,
                        variant="secondary",
                        size="sm"
                    )
                    preset_btns.append((btn, preset_text))
            def set_preset_preset(text):
                return gr.update(value=text)
            # Connect buttons to set positive prompt
            for btn, txt in preset_btns:
                btn.click(fn=set_preset_preset, inputs=[], outputs=prompt_box, _js=None, preprocess=False, postprocess=False, queue=True, kwargs=None, api_name=None, value=txt)
            negative_prompt_tab = gr.Textbox(
                label="Negative Prompt",
                value=default_negative,
                lines=3, max_lines=6,
                elem_id="neg_prompt_box"
            )
        # -------- Video Settings TAB -------- #
        with gr.Tab(label="🎬 Video Settings"):
            with gr.Row():
                width = gr.Number(label="Width", value=480)
                height = gr.Number(label="Height", value=832)
                frames = gr.Number(label="Frames", value=8)
                steps = gr.Number(label="Steps", value=12)
            with gr.Row():
                sampler_name = gr.Dropdown([
                    "uni_pc", "euler", "dpmpp_2m", "ddim", "lms"
                ], value="uni_pc", label="Sampler")
                scheduler = gr.Dropdown([
                    "simple", "normal", "karras", "exponential"
                ], value="simple", label="Scheduler")
                cfg_scale = gr.Number(label="CFG Scale", value=1.0)
                fps = gr.Number(label="FPS", value=12)
                seed = gr.Number(label="Seed (-1 for random)", value=-1)
                output_format = gr.Dropdown(["mp4", "webm"], label="Output Format", value="mp4")
        # -------- LoRAs TAB -------- #
        with gr.Tab(label="🎭 LoRAs"):
            gr.Markdown(
                '<div style="color:#ffb5ea;font-size:1.06em;font-weight:550;">'
                'Paste LoRA(s) you want for style/extra kinks. Multiple accepted, comma/space separated:</div>'
            )
            lora_name = gr.Textbox(label="LoRA Names (.safetensors, CSV)", placeholder="sexy_butt_v12.safetensors,super_giga.safetensors", value="")
            lora_strength = gr.Textbox(label="LoRA Strengths (CSV/space, matches LoRA count)", value="0.7")
            gr.Markdown('<small style="color:#eaa3ff;font-size:1em">LoRA files must be placed in <kbd>/content/ComfyUI/models/lora</kbd> before using. Strength = 0.0~1.5</small>')
        # -------- SPEED/QUALITY TAB -------- #
        with gr.Tab(label="⚡ Speed & Quality"):
            gr.Markdown('<div style="font-size:1.11em;color:#fdb5ff;font-weight:600;margin-bottom:.8em">
            <span style="color:#ffb5ea">Switch speed/quality instantly with a luxury preset.</span></div>')
            with gr.Row():
                preset_dropdown = gr.Radio(
                    choices=list(SPEED_PRESETS.keys()),
                    value="Ultra Fast",
                    label="Speed Preset",
                    interactive=True,
                    elem_id="speed_preset_radio"
                )
                model_quant = gr.Dropdown(["Q4", "Q5", "Q6"], label="Model Quantization (Q4=ultra fast, Q5=best quality)", value="Q4")
                ultra_fast_mode = gr.Checkbox(label="Ultra Fast Mode (preview/T4)", value=True)
            # On selecting preset, autofill settings
            def _update_by_preset(preset):
                mq, w, h, f, st, cfg, smp, sch, uf = speed_preset_apply(preset)
                return mq, uf, w, h, f, st, cfg, smp, sch
            preset_dropdown.change(
                _update_by_preset,
                inputs=preset_dropdown,
                outputs=[model_quant, ultra_fast_mode, width, height, frames, steps, cfg_scale, sampler_name, scheduler]
            )
            gr.Markdown('<small style="color:#fce8ff;font-size:0.97em">Q4 = Fastest (T4/preview), Q5 = Best Quality (A100/4090), Q6 = In Between</small>')
        # -------- OUTPUT TAB (Result + Master Generate) -------- #
        with gr.Tab(label="✨ Generate & View"):
            gr.Markdown('<div style="font-size:1.19em;color:#ffb5ea;font-weight:600;margin-bottom:0.7em">
            Click <b>Generate</b> for an erotic AI video.<br>All settings + LoRA/Prompts are respected.<br></div>')
            gen_btn = gr.Button(
                value="💦 Generate AI Porn Video! 💦",
                elem_id="generate_video_btn",
                variant="primary",
                elem_classes=["gr-button-lg"]
            )
            output = gr.Video(label="Result", interactive=False)

            def _run(
                pos, neg, mq, w, h, f, st, cfg, smp, sch, loran, loras, s, fpsv, fmt, uf
            ):
                return generate_video(
                    positive_prompt=pos,
                    negative_prompt=neg,
                    width=int(w), height=int(h), seed=int(s), steps=int(st), cfg_scale=float(cfg),
                    sampler_name=smp, scheduler=sch, frames=int(f), fps=int(fpsv), lora_name=loran, lora_strength=loras,
                    model_quant=mq, output_format=fmt, ultra_fast_mode=uf
                )

            gen_btn.click(
                _run,
                inputs=[prompt_box, negative_prompt_tab, model_quant, width, height, frames, steps, cfg_scale, sampler_name, scheduler, lora_name, lora_strength, seed, fps, output_format, ultra_fast_mode],
                outputs=output
            )
    # ---- Luxury platform/device info ---- #
    gr.Markdown('''
        <div style="background:linear-gradient(80deg,#190021 50%,#3d0957 100%);margin:2em 0 0 0;padding:1.8em 2em 1em 2em;border-radius:25px;color:#ffe7f8;box-shadow:0 4px 38px #8a21b0a1;font-size:1.08em;">
        <b>Platform Tips:</b><br>
        &bull; <span style="color:#ff6fd8">Ultra Fast (Q4)</span> is best for T4/Free GPUs - ~10s/video.<br>
        &bull; <b>Get better quality or more frames on RTX4090/A100:</b><br><a href='https://lightning.ai' style='color:#ff91ff;text-decoration:underline'><b>Lightning AI (Free A100/4090!)</b></a> or <a href='https://runpod.io/r/city96-wan21' style='color:#ff8efb;text-decoration:underline'><b>RunPod ($0.10/hr)</b></a>.<br>
        &bull; <span style='color:#e9afff'>Copy/paste this notebook to these platforms or run locally. LoRA files: /content/ComfyUI/models/lora</span>
        </div>''')

demo.queue(api_open=False, default_concurrency_limit=2).launch(share=True)
